### Test Image Adjust

In [2]:
# 이미지 한 장 조정 모듈 
"""
입력: 이미지 링크, 저장하고자 하는 위치 

출력: 폴더에 이미지 저장, 

과정
    1. 이미지 링크 가져오기
    2. 링크에서 이미지 로드
    3. 이미지 darkness 정도 파악
    4. darkness 정도에 따라 이미지 조정 파라미터 계산
    5. 계산에 따라 이미지 보정
    6. 보정된 이미지 - 지정된 파일에 저장장

"""

def image_adjust(image_link, save_path):
    pass

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

def get_one_tomatod_3cls_image(split="test", index=0):
    dataset_dir = DATASETS["tomatod_3cls"]
    image_dir = dataset_dir / split / "images"
    images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMG_SUFFIXES])
    return images[index]

def gentle_auto_gamma_bgr(
    image_bgr: np.ndarray,
    target_mean: float = 115.0,
    min_gamma: float = 0.90,
    max_gamma: float = 1.00,
    blend: float = 0.35,
):
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0]
    mean_in = float(y.mean())

    if mean_in >= target_mean:
        gamma = 1.0
        corrected = image_bgr.copy()
    else:
        mean_norm = max(mean_in / 255.0, 1e-6)
        target_norm = target_mean / 255.0
        gamma = np.log(target_norm) / np.log(mean_norm)
        gamma = float(np.clip(gamma, min_gamma, max_gamma))

        lut = np.array(
            [np.clip(((i / 255.0) ** gamma) * 255.0, 0, 255) for i in range(256)],
            dtype=np.uint8,
        )

        out = ycrcb.copy()
        out[:, :, 0] = cv2.LUT(y, lut)
        corrected = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)

    blended = cv2.addWeighted(corrected, blend, image_bgr, 1.0 - blend, 0)

    info = {
        "mean_input": mean_in,
        "gamma": gamma,
        "blend": blend,
        "target_mean": target_mean,
        "mean_output": float(cv2.cvtColor(blended, cv2.COLOR_BGR2YCrCb)[:, :, 0].mean()),
    }
    return blended, info

In [4]:
def shift_tomato_hue(
    image_bgr: np.ndarray,
    # 타겟 Hue 범위 (주황-빨강 토마토)
    hue_low: int = 0,
    hue_high: int = 25,       # H=0~25 범위를 타겟으로
    hue_shift: int = -8,      # 빨강 방향으로 당기기 (음수=빨강 방향)
    sat_boost: float = 1.4,   # 해당 영역 채도 강화
    min_saturation: int = 40, # 너무 무채색은 제외
    min_value: int = 30,      # 너무 어두운 픽셀 제외
) -> np.ndarray:
    
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.int32)
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    
    # 토마토 색상 범위 마스크
    mask = (
        ((h >= hue_low) & (h <= hue_high)) &
        (s >= min_saturation) &
        (v >= min_value)
    )
    
    # Hue 시프트 + 채도 강화
    h[mask] = np.clip(h[mask] + hue_shift, 0, 179)
    s[mask] = np.clip(s[mask] * sat_boost, 0, 255)
    
    hsv[:,:,0] = h
    hsv[:,:,1] = s
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    
    print(f"보정된 픽셀 수: {mask.sum()}")
    return result

In [5]:
def agc_with_selective_color_enhancement(
    image_bgr: np.ndarray,
    expected_intensity: float = 100.0,
    normalize: float = 1.5,
    threshold: float = 0.3,
    sat_scale_low: float = 2.5,
    sat_scale_high: float = 1.1,
    sat_threshold: float = 60.0,
    lab_ab_scale: float = 1.3,
    blend: float = 0.75,
    # ── 추가: 밝기 마스킹 ──────────────────
    brightness_mask: bool = True,       # 어두운 영역만 보정
    bright_pixel_threshold: int = 80,   # 이 값 이상이면 "이미 밝음" → 보정 약하게
    bright_pixel_blend: float = 0.15,   # 밝은 영역에 보정 반영 비율
) -> tuple[np.ndarray, dict]:

    # ── 1. AGC 밝기 보정 ─────────────────────────────────────────────
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0].astype(np.uint8)
    mean_in = float(y.mean())
    t = (mean_in - expected_intensity) / expected_intensity

    if abs(t) <= threshold:
        brightness_corrected = image_bgr.copy()
    else:
        reversed_mode = t > threshold
        work_y = 255 - y if reversed_mode else y

        hist = cv2.calcHist([work_y], [0], None, [256], [0, 256]).flatten()
        prob = hist / hist.sum()
        prob_min, prob_max = prob.min(), prob.max()

        pn_temp = np.zeros_like(prob, dtype=np.float64)
        mask = prob > 0
        pn_temp[mask] = ((prob[mask] - prob_min) / (prob_max - prob_min)) ** normalize
        prob_wd = pn_temp / pn_temp.sum() * normalize
        cdf_wd = np.cumsum(prob_wd)
        inverse_cdf = normalize - cdf_wd

        lut = np.array([
            np.clip(round(255 * ((i / 255.0) ** inverse_cdf[i])), 0, 255)
            for i in range(256)
        ], dtype=np.uint8)

        corrected_y = lut[work_y]
        if reversed_mode:
            corrected_y = 255 - corrected_y

        if brightness_mask:
            # 밝은 픽셀은 보정을 거의 안 받도록 픽셀별 블렌딩
            dark_mask = (y < bright_pixel_threshold).astype(np.float32)  # 0 or 1
            # 어두운 픽셀: 100% 보정 / 밝은 픽셀: bright_pixel_blend만 보정
            blend_map = dark_mask + (1 - dark_mask) * bright_pixel_blend
            corrected_y = (corrected_y * blend_map + y * (1 - blend_map)).astype(np.uint8)

        out = ycrcb.copy()
        out[:, :, 0] = corrected_y
        brightness_corrected = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)

    # ── 2. 채도 선택적 강화 ───────────────────────────────────────────
    hsv = cv2.cvtColor(brightness_corrected, cv2.COLOR_BGR2HSV).astype(np.float32)
    s = hsv[:, :, 1]
    scale_map = np.where(s < sat_threshold, sat_scale_low, sat_scale_high)
    hsv[:, :, 1] = np.clip(s * scale_map, 0, 255)
    sat_enhanced = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # ── 3. LAB 색 대비 강화 ───────────────────────────────────────────
    lab = cv2.cvtColor(sat_enhanced, cv2.COLOR_BGR2LAB).astype(np.float32)
    
    # a채널(빨강-초록): 128 중심으로 스케일 → 빨강은 더 빨갛게, 초록은 더 초록으로
    a = lab[:, :, 1]
    lab[:, :, 1] = np.clip((a - 128) * lab_ab_scale + 128, 0, 255)
    
    # b채널(노랑-파랑): 동일하게 128 중심
    b = lab[:, :, 2]
    lab[:, :, 2] = np.clip((b - 128) * lab_ab_scale + 128, 0, 255)
    
    color_enhanced = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
    
    # ── 3.5 토마토 Hue 보정 ──────────────────────────────────────────
    
    color_enhanced = shift_tomato_hue(
        color_enhanced,
        hue_low=0,
        hue_high=50,
        hue_shift=-10,
        sat_boost=1.5,
        min_saturation=85,   # 35 → 80 (흙/갈색 제외)
        min_value=60,        # 25 → 60 (너무 어둡거나 탁한 픽셀 제외)
    )
    
    # ── 4. 블렌딩 ─────────────────────────────────────────────────────
    result = cv2.addWeighted(color_enhanced, blend, image_bgr, 1.0 - blend, 0)

    info = {
        "mean_input": mean_in,
        "mean_output": float(cv2.cvtColor(result, cv2.COLOR_BGR2YCrCb)[:, :, 0].mean()),
        "sat_scale_low": sat_scale_low,
        "sat_scale_high": sat_scale_high,
        "sat_threshold": sat_threshold,
        "lab_ab_scale": lab_ab_scale,
        "blend": blend,
        "bright_pixel_threshold": bright_pixel_threshold,
        "bright_pixel_blend": bright_pixel_blend,
    }
    return result, info

In [6]:
def agc_tomato_focused(
    image_bgr: np.ndarray,
    # AGC
    expected_intensity: float = 90.0,
    normalize: float = 1.5,
    threshold: float = 0.3,
    bright_pixel_threshold: int = 80,
    bright_pixel_blend: float = 0.15,
    # 토마토 색상 타겟 보정만
    tomato_hue_low: int = 0,
    tomato_hue_high: int = 50,
    tomato_sat_boost: float = 1.8,
    tomato_min_sat: int = 80,
    tomato_min_val: int = 60,
    tomato_hue_shift: int = -5,
    # 블렌딩
    blend: float = 0.65,
) -> tuple[np.ndarray, dict]:

    # ── 1. AGC 밝기 보정 ─────────────────────────────────────────────
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0].astype(np.uint8)
    mean_in = float(y.mean())
    t = (mean_in - expected_intensity) / expected_intensity

    if abs(t) <= threshold:
        brightness_corrected = image_bgr.copy()
    else:
        reversed_mode = t > threshold
        work_y = 255 - y if reversed_mode else y

        hist = cv2.calcHist([work_y], [0], None, [256], [0, 256]).flatten()
        prob = hist / hist.sum()
        prob_min, prob_max = prob.min(), prob.max()
        pn_temp = np.zeros_like(prob, dtype=np.float64)
        mask = prob > 0
        pn_temp[mask] = ((prob[mask] - prob_min) / (prob_max - prob_min)) ** normalize
        prob_wd = pn_temp / pn_temp.sum() * normalize
        inverse_cdf = normalize - np.cumsum(prob_wd)

        lut = np.array([
            np.clip(round(255 * ((i / 255.0) ** inverse_cdf[i])), 0, 255)
            for i in range(256)
        ], dtype=np.uint8)

        corrected_y = lut[work_y]
        if reversed_mode:
            corrected_y = 255 - corrected_y

        # 밝은 픽셀은 보정 최소화
        dark_mask = (y < bright_pixel_threshold).astype(np.float32)
        blend_map = dark_mask + (1 - dark_mask) * bright_pixel_blend
        corrected_y = (corrected_y * blend_map + y * (1 - blend_map)).astype(np.uint8)

        out = ycrcb.copy()
        out[:, :, 0] = corrected_y
        brightness_corrected = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)

    # ── 2. 토마토 Hue 타겟 보정만 ────────────────────────────────────
    hsv = cv2.cvtColor(brightness_corrected, cv2.COLOR_BGR2HSV).astype(np.int32)
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]

    tomato_mask = (
        (h >= tomato_hue_low) & (h <= tomato_hue_high) &
        (s >= tomato_min_sat) &
        (v >= tomato_min_val)
    )
    h[tomato_mask] = np.clip(h[tomato_mask] + tomato_hue_shift, 0, 179)
    s[tomato_mask] = np.clip(s[tomato_mask] * tomato_sat_boost, 0, 255)

    hsv[:,:,0] = h
    hsv[:,:,1] = s
    color_corrected = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # ── 3. 블렌딩 ─────────────────────────────────────────────────────
    result = cv2.addWeighted(color_corrected, blend, image_bgr, 1.0 - blend, 0)

    info = {
        "mean_input": mean_in,
        "mean_output": float(cv2.cvtColor(result, cv2.COLOR_BGR2YCrCb)[:, :, 0].mean()),
        "tomato_pixels_corrected": int(tomato_mask.sum()),
        "tomato_sat_boost": tomato_sat_boost,
        "tomato_hue_shift": tomato_hue_shift,
        "blend": blend,
    }
    return result, info

In [7]:
img_path = get_one_tomatod_3cls_image(split="test", index=30)
img_bgr = cv2.imread(str(img_path))

result_bgr, info = agc_with_selective_color_enhancement(
    img_bgr,
    expected_intensity=100.0,
    sat_scale_low=2.0,
    sat_scale_high=1.0,
    sat_threshold=60.0,
    lab_ab_scale=1.2,
    blend=0.7,
)

print("image:", img_path.name)
for k, v in info.items():
    print(f"{k}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(
    f"AGC + Selective Color Enhanced\n"
    f"mean {info['mean_input']:.1f} → {info['mean_output']:.1f} | "
    f"sat_low×{info['sat_scale_low']} sat_high×{info['sat_scale_high']} | "
    f"lab_ab×{info['lab_ab_scale']}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

NameError: name 'DATASETS' is not defined

In [ ]:
result_bgr, info = agc_tomato_focused(img_bgr)

print("image:", img_path.name)
for k, v in info.items():
    print(f"{k}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(
    f"AGC + Tomato Focused\n"
    f"mean {info['mean_input']:.1f} → {info['mean_output']:.1f} | "
    f"tomato_px={info['tomato_pixels_corrected']} | "
    f"sat×{info['tomato_sat_boost']}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

### semi-ripe 색상 강조 방법 비교

zero-shot VLM 분류에서 semi-ripe(빨강-주황-초록 혼재) crop이 fully-ripe/unripe로
잘못 판정되는 경우가 많아서, semi-ripe 특유의 색 전이가 더 잘 보이도록 crop 자체를
보정하는 여러 방법을 실제 semi-ripe crop에 적용해서 나란히 비교한다.

DATASETS 딕셔너리가 정의되어 있지 않아 위 get_one_tomatod_3cls_image()는 실행이
안 되므로, 대신 프로젝트에 이미 있는 COCO GT 로더(src.vlm.dataset.load_coco_records)
+ crop 로더(src.vlm.cropping.load_crop)를 그대로 재사용해서 실제 semi-ripe crop을
가져온다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebook").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.vlm.cropping import load_crop
from src.vlm.dataset import load_coco_records

def load_sample_crops_bgr(dataset_name="tomatod_3cls", split="test", class_name="semi-ripe", n=6, expand_ratio=0.1, min_size=32):
    """GT COCO 라벨에서 지정한 클래스의 crop을 n개 골라 (record, crop_bgr) 리스트로 반환한다."""
    coco_root = PROJECT_ROOT / "data" / "coco" / dataset_name
    records, _ = load_coco_records(coco_root, split, class_map=CANONICAL_CLASS_ID_TO_NAME)
    matched = [record for record in records if record["class_name"] == class_name]
    samples = []
    for record in matched[:n]:
        crop_rgb = load_crop(record, expand_ratio=expand_ratio, min_size=min_size)
        crop_bgr = cv2.cvtColor(np.array(crop_rgb), cv2.COLOR_RGB2BGR)
        samples.append((record, crop_bgr))
    return samples

from src.vlm.schema import CANONICAL_CLASS_ID_TO_NAME

semi_ripe_samples = load_sample_crops_bgr(dataset_name="tomatod_3cls", split="test", class_name="semi-ripe", n=6)
print(f"semi-ripe crop {len(semi_ripe_samples)}개 로드")
for record, crop_bgr in semi_ripe_samples:
    print(record["sample_id"], crop_bgr.shape)

In [ ]:
def apply_saturation_boost_bgr(image_bgr: np.ndarray, factor: float = 1.4) -> np.ndarray:
    """HSV 채도(S) 채널만 전역적으로 factor배 하는 가장 단순한 방법.

    빨강/주황/초록이 전체적으로 더 진해져서 서로 구분이 쉬워지지만, 이미 채도가
    높은 영역(과다노출된 빨강 등)은 clip되어 디테일이 뭉개질 수 있다.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def apply_clahe_color_bgr(
    image_bgr: np.ndarray,
    *,
    clip_limit: float = 2.0,
    tile_grid_size: tuple[int, int] = (8, 8),
    apply_to_ab: bool = True,
) -> np.ndarray:
    """CLAHE를 LAB의 L(명도) 채널뿐 아니라 a/b(색상) 채널까지 적용한다.

    src/vlm/image_adjustment.py의 apply_clahe는 L 채널만 보정한다(명도 대비만
    개선, 색상 분포는 그대로) - 여기서는 a(빨강-초록)/b(노랑-파랑) 채널에도
    CLAHE를 걸어서, 국소적으로 색이 섞여 있는 영역(semi-ripe 전이부)의 색 대비
    자체를 끌어올린다.
    """
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_channel, a_channel, b_channel = cv2.split(lab)
    l_channel = clahe.apply(l_channel)
    if apply_to_ab:
        a_channel = clahe.apply(a_channel)
        b_channel = clahe.apply(b_channel)
    merged = cv2.merge([l_channel, a_channel, b_channel])
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

In [ ]:
METHODS = {
    "Original": lambda img: img,
    "Saturation x1.4 (fixed)": lambda img: apply_saturation_boost_bgr(img, factor=1.4),
    "Adaptive saturation": lambda img: apply_adaptive_saturation_boost_bgr(img)[0],
    "Adaptive brightness+saturation": lambda img: apply_adaptive_brightness_saturation_bgr(img)[0],
    "CLAHE (L only)": lambda img: apply_clahe_color_bgr(img, apply_to_ab=False),
    "shift_tomato_hue": lambda img: shift_tomato_hue(img),
    "agc_tomato_focused": lambda img: agc_tomato_focused(img)[0],
}

n_samples = len(semi_ripe_samples)
n_methods = len(METHODS)
fig, axes = plt.subplots(n_samples, n_methods, figsize=(3 * n_methods, 3 * n_samples))
if n_samples == 1:
    axes = axes[None, :]

for row, (record, crop_bgr) in enumerate(semi_ripe_samples):
    for col, (method_name, method_fn) in enumerate(METHODS.items()):
        result_bgr = method_fn(crop_bgr)
        ax = axes[row, col]
        ax.imshow(cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB))
        if row == 0:
            ax.set_title(method_name, fontsize=9)
        if col == 0:
            ax.set_ylabel(record["sample_id"], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("semi-ripe crop 색상 강조 방법 비교 (GT class = semi-ripe)", y=1.0)
plt.tight_layout()
plt.show()

### adaptive 버전 - 이미지마다 자기 통계로 보정 강도를 정함

위 방법들은 factor/threshold가 전부 고정값이라, 원래 진한 crop엔 과하게 걸리고
원래 흐린 crop엔 부족하게 걸릴 수 있다. gentle_auto_gamma_bgr가 이미
각 이미지의 평균 밝기(mean_in)를 재서 목표 밝기(target_mean)에 맞는 gamma를
그때그때 계산하는 방식이므로, 채도도 같은 방식(각 crop의 평균 채도를 재서
목표 채도에 맞는 배율을 그 이미지에 맞게 계산)으로 만든다.

In [ ]:
def apply_adaptive_saturation_boost_bgr(
    image_bgr: np.ndarray,
    *,
    target_mean_saturation: float = 130.0,
    min_factor: float = 1.0,
    max_factor: float = 2.5,
) -> tuple[np.ndarray, dict]:
    """crop 자신의 평균 채도를 재서, 목표 채도(target_mean_saturation)에 맞는
    배율을 그때그때 계산해 적용한다 (고정 factor=1.4를 쓰던 apply_saturation_boost_bgr와
    달리 이미지마다 다른 배율이 걸린다).

    이미 채도가 높은 crop은 factor가 1.0에 가깝게 나와서 거의 안 건드리고,
    저채도 crop은 max_factor까지 세게 걸린다.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    mean_sat_in = float(hsv[:, :, 1].mean())
    factor = target_mean_saturation / max(mean_sat_in, 1e-6)
    factor = float(np.clip(factor, min_factor, max_factor))
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * factor, 0, 255)
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    info = {
        "mean_saturation_in": mean_sat_in,
        "factor": factor,
        "target_mean_saturation": target_mean_saturation,
        "mean_saturation_out": float(cv2.cvtColor(result, cv2.COLOR_BGR2HSV)[:, :, 1].mean()),
    }
    return result, info


def apply_adaptive_brightness_saturation_bgr(
    image_bgr: np.ndarray,
    *,
    target_mean_brightness: float = 120.0,
    target_mean_saturation: float = 130.0,
    min_gamma: float = 0.55,
    max_gamma: float = 1.0,
    max_saturation_factor: float = 2.5,
) -> tuple[np.ndarray, dict]:
    """gentle_auto_gamma_bgr(밝기, blend=1.0으로 전체 반영) 다음에
    apply_adaptive_saturation_boost_bgr(채도)를 순서대로 적용한다.

    semi-ripe crop이 원래 어두운 채로 채도만 올리면 거의 안 바뀐다는 게 이전
    비교에서 확인됐기 때문에 - 어둡던 crop을 먼저 목표 밝기까지 끌어올린 다음에
    채도를 보정해야 색이 실제로 드러난다. hue는 건드리지 않는다 (특정 색으로
    미는 게 아니라 원래 색 대비만 살리는 목적).
    """
    brightened, brightness_info = gentle_auto_gamma_bgr(
        image_bgr,
        target_mean=target_mean_brightness,
        min_gamma=min_gamma,
        max_gamma=max_gamma,
        blend=1.0,
    )
    result, saturation_info = apply_adaptive_saturation_boost_bgr(
        brightened,
        target_mean_saturation=target_mean_saturation,
        max_factor=max_saturation_factor,
    )
    info = {**brightness_info, **saturation_info}
    return result, info

NameError: name 'np' is not defined